<h2> Feature selection - Chi and Anova </h2>

In [19]:
from sklearn.model_selection import train_test_split
import pandas as pd

df_normalized = pd.read_csv('booster_normalized.csv') # change csv file name
target_normalized = df_normalized['AgreeSubsequentBooster']


features_normalized = df_normalized.drop('AgreeSubsequentBooster', axis=1)

x_train_normalized, x_test_normalized, y_train_normalized, y_test_normalized = train_test_split(features_normalized,
                                                                                                target_normalized,
                                                                                                test_size=0.2,
                                                                                                random_state=0)

In [20]:
anova_columns = [
    'Age','EmploymentYears',

    'A_mean', 'B_mean', 'C_mean', 'D_mean', 'E_mean'
]

chi2_columns = [
    'Gender','Education','Income','Pregnant','PatientContact','Occupation',
    'CovidPatientCare','PreExistingCondition','CovidInfected','Severity',
    'AdditionalVaccines','VaccinationStage','VaccinationSideEffects',
    'FamilySideEffects','SideEffectsAffectView',

    # one-hot race dummies
    'Race_Chinese','Race_Indian','Race_Malay','Race_Siam',

    # one-hot religion dummies
    'Religion_Buddha','Religion_Hindu','Religion_Islam',

    # one-hot location dummies
    'Location_Johor','Location_Kedah','Location_Kelantan','Location_Kuantan ',
    'Location_Melaka','Location_Negeri Sembilan','Location_Pahang',
    'Location_Perlis','Location_Pulau Pinang','Location_Sabah',
    'Location_Selangor','Location_Terengganu'
]


# Extract and save ANOVA columns from the training set
anova_train = x_train_normalized[anova_columns]

# Extract and save Chi-Squared columns from the training set
chi2_train = x_train_normalized[chi2_columns]

<h2>Anova testing</h2>

In [21]:
import pandas as pd
# Import SelectKBest and chi2 modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif


# Apply SelectKBest with ANOVA on the anova_train data
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(anova_train, y_train_normalized)
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': anova_train.columns,
    'ANOVA Score': scores,
    'P-value': p_values
})

# Sort the DataFrame by ANOVA scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='ANOVA Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

           Feature  ANOVA Score   P-value
4           C_mean    25.738069  0.000001
2           A_mean    21.085981  0.000009
6           E_mean    12.942599  0.000430
5           D_mean    11.085383  0.001084
1  EmploymentYears     9.016271  0.003115
0              Age     4.341839  0.038807
3           B_mean     0.026584  0.870692


In [22]:
# Select features with p-value < 0.05 # 0.05 is the default value. less than 0.05 is better
significant_features_anova = feature_scores[feature_scores['P-value'] < 0.05]

# Display significant features
print(significant_features_anova)

           Feature  ANOVA Score   P-value
0              Age     4.341839  0.038807
1  EmploymentYears     9.016271  0.003115
2           A_mean    21.085981  0.000009
4           C_mean    25.738069  0.000001
5           D_mean    11.085383  0.001084
6           E_mean    12.942599  0.000430


<h2>Chi2 Testing</h2>

In [23]:
# Import SelectKBest and chi2 modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2


# Apply SelectKBest with chi2
selector = SelectKBest(score_func=chi2, k='all')
selector.fit(chi2_train, y_train_normalized)
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': chi2_train.columns,
    'Chi-squared Score': scores,
    'P-value': p_values
})

# Sort the DataFrame by Chi-squared scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='Chi-squared Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                     Feature  Chi-squared Score   P-value
14     SideEffectsAffectView          13.275127  0.000269
1                  Education           5.062757  0.024445
15              Race_Chinese           3.395349  0.065381
13         FamilySideEffects           3.376872  0.066117
10        AdditionalVaccines           2.331641  0.126768
16               Race_Indian           1.697674  0.192592
32         Location_Selangor           1.697674  0.192592
20            Religion_Hindu           1.697674  0.192592
8              CovidInfected           1.406773  0.235593
6           CovidPatientCare           1.367752  0.242198
19           Religion_Buddha           1.351896  0.244947
33       Location_Terengganu           1.351896  0.244947
3                   Pregnant           1.351896  0.244947
31            Location_Sabah           1.178082  0.277747
18                 Race_Siam           1.178082  0.277747
25         Location_Kuantan            1.178082  0.277747
0             

In [24]:
# Select features with p-value < 0.05
significant_features_chi = feature_scores[feature_scores['P-value'] < 0.05]

# Display significant features
print(significant_features_chi)

                  Feature  Chi-squared Score   P-value
1               Education           5.062757  0.024445
14  SideEffectsAffectView          13.275127  0.000269


<h2>Information Gain</h2>

In [25]:
# Import SelectKBest and information gain modules
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

# Apply SelectKBest with information gain
selector = SelectKBest(score_func=mutual_info_classif, k='all')
selector.fit(chi2_train, y_train_normalized)
scores = selector.scores_

# Create a DataFrame to store feature names, scores, and p-values
feature_scores = pd.DataFrame({
    'Feature': chi2_train.columns,
    'Information gain Score': scores,
})

# Sort the DataFrame by Chi-squared scores in descending order
feature_scores_sorted = feature_scores.sort_values(by='Information gain Score', ascending=False)

# Display the DataFrame
print(feature_scores_sorted)

                     Feature  Information gain Score
21            Religion_Islam                0.107568
1                  Education                0.077365
4             PatientContact                0.058890
18                 Race_Siam                0.047852
2                     Income                0.047748
20            Religion_Hindu                0.046728
14     SideEffectsAffectView                0.038911
15              Race_Chinese                0.026097
31            Location_Sabah                0.024874
30     Location_Pulau Pinang                0.016637
28           Location_Pahang                0.014875
0                     Gender                0.013701
27  Location_Negeri Sembilan                0.012422
11          VaccinationStage                0.007006
25         Location_Kuantan                 0.000000
24         Location_Kelantan                0.000000
23            Location_Kedah                0.000000
29           Location_Perlis                0.

In [26]:
selected_features_in = feature_scores_sorted[feature_scores_sorted['Information gain Score'] > 0.05]
print(selected_features_in)

           Feature  Information gain Score
21  Religion_Islam                0.107568
1        Education                0.077365
4   PatientContact                0.058890


<h2>Wrapper approach</h2>

<h2>RFE</h2>

In [27]:
# Import necessary libraries
from sklearn.feature_selection import RFE
from sklearn.svm import SVC

# Create an SVM classifier with a linear kernel
svm_linear = SVC(kernel='linear')

# Initialize RFE with the SVM classifier
rfe = RFE(estimator=svm_linear, step=1)

# Fit RFE on the training data
rfe.fit(x_train_normalized, y_train_normalized)

# Print the results
print("Number of Features: ", rfe.n_features_)
print("Feature Ranking: ", rfe.ranking_)
print("Selected Features: ", rfe.support_)

# Print the indices of the selected features
selected_feature_indices = [i for i, x in enumerate(rfe.support_) if x]
print("Selected Feature Indices: ", selected_feature_indices)

Number of Features:  20
Feature Ranking:  [ 1  1  1  1  4  1 13  7 11 10 12 17  1 19  1  3  1  1  1 14  1  5  1 15
 20  1  8  1 21  1  1 22  1  1 18  2  6  9  1  1 16]
Selected Features:  [ True  True  True  True False  True False False False False False False
  True False  True False  True  True  True False  True False  True False
 False  True False  True False  True  True False  True  True False False
 False False  True  True False]
Selected Feature Indices:  [0, 1, 2, 3, 5, 12, 14, 16, 17, 18, 20, 22, 25, 27, 29, 30, 32, 33, 38, 39]


In [28]:
# Get the names of the selected features
selected_feature_names = x_train_normalized.columns[rfe.support_]
print("Selected Feature Names:")
for feature_name in selected_feature_names.tolist():
    print(feature_name)

Selected Feature Names:
Age
Gender
Education
Income
PatientContact
AdditionalVaccines
VaccinationSideEffects
SideEffectsAffectView
Race_Chinese
Race_Indian
Race_Siam
Religion_Hindu
Location_Kedah
Location_Kuantan 
Location_Negeri Sembilan
Location_Pahang
Location_Pulau Pinang
Location_Sabah
C_mean
D_mean


<h2>OUTPUT</h2>

In [29]:
result1 = pd.concat([significant_features_anova,significant_features_chi])
result1 = result1.drop(columns=['ANOVA Score', 'P-value', 'Chi-squared Score'])
print (result1)

                  Feature
0                     Age
1         EmploymentYears
2                  A_mean
4                  C_mean
5                  D_mean
6                  E_mean
1               Education
14  SideEffectsAffectView


In [30]:
significant_features = result1['Feature'].tolist()
anova_chi = df_normalized[significant_features + ['AgreeSubsequentBooster']]
anova_chi.to_csv('anova_chi_normalized.csv', index=False) # 200 features
# anova_chi.to_csv('ac_350_test.csv') # 350 features

In [31]:
# anova_chi_not_normalized = df_not_normalized[significant_features + ['AgreeSubsequentBooster']]
# anova_chi_not_normalized.to_csv('anova_chi_not_normalized.csv', index=False)

In [32]:
result2 = pd.concat([significant_features_anova,selected_features_in])
result2 = result2.drop(columns=['ANOVA Score', 'P-value', 'Information gain Score'])
print (result2)

            Feature
0               Age
1   EmploymentYears
2            A_mean
4            C_mean
5            D_mean
6            E_mean
21   Religion_Islam
1         Education
4    PatientContact


In [33]:
significant_features = result2['Feature'].tolist()
anova_in = df_normalized[significant_features + ['AgreeSubsequentBooster']]
anova_in.to_csv('anova_cin_normalized.csv', index=False)

In [34]:
# anova_in_not_normalized = df_not_normalized[significant_features + ['AgreeSubsequentBooster']]
# anova_in_not_normalized.to_csv('anova_cin_not_normalized.csv', index=False)

In [35]:
result3 = pd.concat([significant_features_chi,selected_features_in])
result3 = result3.drop(columns=['Chi-squared Score', 'P-value', 'Information gain Score'])
print (result3)

                  Feature
1               Education
14  SideEffectsAffectView
21         Religion_Islam
1               Education
4          PatientContact


In [36]:
significant_features = result3['Feature'].tolist()
chi_in = df_normalized[significant_features + ['AgreeSubsequentBooster']]
chi_in.to_csv('chi_in.csv', index=False)